In [1]:
# Nessary libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import os

os.getcwd()

'E:\\Imarticus_ Learning\\Jobs - 2026\\Models\\notebooks\\EDA in python'

In [4]:
path = "../../data/Processed/phishing_websites_dataset"

In [35]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
phishing_websites = fetch_ucirepo(id=327) 
  
# data (as pandas dataframes) 
X = phishing_websites.data.features 
y = phishing_websites.data.targets 



X.to_csv(f"{path}/X.csv", index=False)
y.to_csv(f"{path}/y.csv", index=False)

In [5]:
X = pd.read_csv(path + "/X.csv")
y = pd.read_csv(path + "/y.csv")
print(X.shape)
print(y.shape)

(11055, 30)
(11055, 1)


In [6]:
# Data Cleaning and Imputation
# Addressing the missing values
null_cols_X = X.columns[X.isna().sum() > 0]
print(null_cols_X)

Index([], dtype='object')


In [7]:
null_values_y = y.isna().sum()
print(null_values_y)

result    0
dtype: int64


# Addressing the Missig Data
### Here there are no missing values so this step might be skipped, if there missing values we might use these stratgies
- Drop missing values if they are from the target variable. and Drop missing values if there are overall less than 5% of the df
- Impute using mean, median and mode depending the distribution and context
- Impute by sub groups , if there is a relevant categorical column.
  - to do that we need sub_groups = df.groupby['Cat_column']('numeric_column').median() #Here I have used median but we can use any central tendency method#  and convert it into a dictonary using the subs_groups = df.to_dict() then using map function we can fill those missing values with that dict. Using this code df['null_column'] = df['null_colum'].fillna(df['cat_column']).map(sub_groups))

In [8]:
X.columns

Index(['having_ip_address', 'url_length', 'shortining_service',
       'having_at_symbol', 'double_slash_redirecting', 'prefix_suffix',
       'having_sub_domain', 'sslfinal_state', 'domain_registration_length',
       'favicon', 'port', 'https_token', 'request_url', 'url_of_anchor',
       'links_in_tags', 'sfh', 'submitting_to_email', 'abnormal_url',
       'redirect', 'on_mouseover', 'rightclick', 'popupwindow', 'iframe',
       'age_of_domain', 'dnsrecord', 'web_traffic', 'page_rank',
       'google_index', 'links_pointing_to_page', 'statistical_report'],
      dtype='object')

In [9]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11055 entries, 0 to 11054
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   having_ip_address           11055 non-null  int64
 1   url_length                  11055 non-null  int64
 2   shortining_service          11055 non-null  int64
 3   having_at_symbol            11055 non-null  int64
 4   double_slash_redirecting    11055 non-null  int64
 5   prefix_suffix               11055 non-null  int64
 6   having_sub_domain           11055 non-null  int64
 7   sslfinal_state              11055 non-null  int64
 8   domain_registration_length  11055 non-null  int64
 9   favicon                     11055 non-null  int64
 10  port                        11055 non-null  int64
 11  https_token                 11055 non-null  int64
 12  request_url                 11055 non-null  int64
 13  url_of_anchor               11055 non-null  int64
 14  links_

# Converting and Analyzing categorical data

In [10]:
X.select_dtypes('int').head() # to view the category columns and first five records 

,having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,...,rightclick,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,1,-1,-1,-1,-1,1,1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,1,-1,-1,0,-1,1,1,1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,1,-1,1,-1,1,0,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,1,-1,-1,1,-1,1,-1,1
4,1,0,-1,1,1,-1,1,1,-1,1,...,1,-1,1,-1,-1,0,-1,1,1,1


##### if we want to see what are the categorical columns available we can use the above code. but some times there might be already encoded cat columns which might be a int type so what we can do is we can use .nunique using for each columns and decide whether it is cat column or numerical column.

In [11]:
"""# Lowercase all the values so that it can be easy to filter
# filtering single phrases in the string of a cat column
df['cat_column'].str.contains('Exactword')
# filtering multiple phrases in the string of a cat column
df['cat_column'].str.contains('Exactword|anotherword') # lookout for the spaces
# filtering that starts with that word
df['cat_column'].str.contains('^startswith')"""

"# Lowercase all the values so that it can be easy to filter\n# filtering single phrases in the string of a cat column\ndf['cat_column'].str.contains('Exactword')\n# filtering multiple phrases in the string of a cat column\ndf['cat_column'].str.contains('Exactword|anotherword') # lookout for the spaces\n# filtering that starts with that word\ndf['cat_column'].str.contains('^startswith')"

In [12]:
"""# Say we have multiple job categories but we want to bin them into certain category to do that 
# Derive what are the job categories we need 
job_categories = [ 'Data Science', 'Data Analytics', 'Data Engineering', 'Machine Learning', 'Managerial', 'Consultant']
# Create variables containign the filters 
data_science = "data scientist|nlp"
data_analyst - "analyst|analytics"
data_engineer = "data enginerr|etl|architect|infrastructure"
ml_engineer = "machine learnig|ml|big data|ai"
manager = "manager|head|lead|principal|staff
consultant = "consultant|freelance"
# Create a list with range of condtions
conditions = [
    (salaries['Designation'].str.contains(data_science)),
    (salaries['Designation'].str.contains(data_analyst)),
    (salaries['Designation'].str.contains(data_engineer)),
    (salaries['Designation'].str.contains(ml_engineer)),
    (salaries['Designation'].str.contains(manager)),
    (salaries['Designation'].str.contains(consultant))
]
# Create a new cat column with job category
salaries['Job_Category'] = np.select(conditions,
                                     job_categories,
                                     default = 'Others')
# Visualize each category
sns.countplot(data = df, x = 'job_category')
plt.show()"""

'# Say we have multiple job categories but we want to bin them into certain category to do that \n# Derive what are the job categories we need \njob_categories = [ \'Data Science\', \'Data Analytics\', \'Data Engineering\', \'Machine Learning\', \'Managerial\', \'Consultant\']\n# Create variables containign the filters \ndata_science = "data scientist|nlp"\ndata_analyst - "analyst|analytics"\ndata_engineer = "data enginerr|etl|architect|infrastructure"\nml_engineer = "machine learnig|ml|big data|ai"\nmanager = "manager|head|lead|principal|staff\nconsultant = "consultant|freelance"\n# Create a list with range of condtions\nconditions = [\n    (salaries[\'Designation\'].str.contains(data_science)),\n    (salaries[\'Designation\'].str.contains(data_analyst)),\n    (salaries[\'Designation\'].str.contains(data_engineer)),\n    (salaries[\'Designation\'].str.contains(ml_engineer)),\n    (salaries[\'Designation\'].str.contains(manager)),\n    (salaries[\'Designation\'].str.contains(consultant

In [13]:

#Finding the number of unique values
# Filter the DataFrame for object columns
non_numeric = X.select_dtypes("int")

# Loop through columns
for i in non_numeric.columns:
  
  # Print the number of unique values
  print(f"Number of unique values in {i} column: ", non_numeric[i].nunique())

Number of unique values in having_ip_address column:  2
Number of unique values in url_length column:  3
Number of unique values in shortining_service column:  2
Number of unique values in having_at_symbol column:  2
Number of unique values in double_slash_redirecting column:  2
Number of unique values in prefix_suffix column:  2
Number of unique values in having_sub_domain column:  3
Number of unique values in sslfinal_state column:  3
Number of unique values in domain_registration_length column:  2
Number of unique values in favicon column:  2
Number of unique values in port column:  2
Number of unique values in https_token column:  2
Number of unique values in request_url column:  2
Number of unique values in url_of_anchor column:  3
Number of unique values in links_in_tags column:  3
Number of unique values in sfh column:  3
Number of unique values in submitting_to_email column:  2
Number of unique values in abnormal_url column:  2
Number of unique values in redirect column:  2
Num

# Working with numeric data

##### Here there are no numeric data and whe have to check the distribution and handle the outliers
- Using IQR handle the outliers
- identify the thesehold
- identufy the outliers and retun the outlier data using subsetting the data
- if present
  - ask why they exists?
  - is the data accurate error values then drop it
  - if valid outliers using transfornmation.